In [ ]:
import pennylane as qml
from pennylane.operation import Operation
import numpy as np

class RBS(Operation):
    num_wires = 2
    num_params = 1
    par_domain = "R"

    def __init__(self, theta, wires):
        super().__init__(theta, wires=wires)

    @staticmethod
    def compute_matrix(theta):
        c = np.cos(theta)
        s = np.sin(theta)

        return np.array([
            [1, 0,  0, 0],
            [0, c,  s, 0],
            [0, -s, c, 0],
            [0, 0,  0, 1]
        ])


In [ ]:
def butterfly_rbs_ansatz(n_qubits, n_layers):
    assert (n_qubits & (n_qubits - 1)) == 0, "n_qubits must be power of 2"
    
    n_stages = int(np.log2(n_qubits))
    thetas = []

    for layer in range(n_layers):
        for stage in range(n_stages):
            d = 2 ** stage
            for i in range(n_qubits):
                if i % (2 * d) < d:
                    theta = qml.numpy.array(0.0, requires_grad=True)
                    RBS(theta, wires=[i, i + d])
                    thetas.append(theta)

    return thetas


In [ ]:
dev = qml.device("default.qubit", wires=8)

@qml.qnode(dev)
def circuit():
    butterfly_rbs_ansatz(n_qubits=8, n_layers=1)
    return qml.state()

print(qml.draw(circuit)())


## Inverted

In [ ]:
import numpy as np
import pennylane as qml

def reverse_butterfly_rbs_ansatz(n_qubits, n_layers):
    assert (n_qubits & (n_qubits - 1)) == 0, "n_qubits must be power of 2"

    n_stages = int(np.log2(n_qubits))
    print ("n_stages",n_stages)
    thetas = []
    print("aqui n_layers",n_layers)
    for layer in range(n_layers):
        print ("---layer",layer)
        for stage in reversed(range(n_stages)):
            print("stage",stage)
            d = 2 ** stage
            for i in range(n_qubits):
                if i % (2 * d) < d:
                    print("stage",stage,"i",i,"| d",d,"| i + d",i + d)
                    theta = qml.numpy.array(0.0, requires_grad=True)
                    RBS(theta, wires=[i, i + d])
                    thetas.append(theta)

    return thetas


In [ ]:
dev = qml.device("default.qubit", wires=8)

@qml.qnode(dev)
def circuit():
    reverse_butterfly_rbs_ansatz(n_qubits=8, n_layers=1)
    return qml.state()

print(qml.draw(circuit)())


## N=16

In [ ]:
import pennylane as qml
import numpy as np

"""
class RBS(qml.operation.Operation):
    num_wires = 2
    num_params = 1
    grad_method = "A"
    name = "RBS"

    @staticmethod
    def decomposition(theta, wires):
        return [
            qml.CNOT(wires=wires),
            qml.RY(theta, wires=wires[0]),
            qml.RY(-theta, wires=wires[1]),
            qml.CNOT(wires=wires),
        ]

    def label(self, decimals=None, base_label=None):
        theta = qml.math.toarray(self.parameters[0]).item()
        theta = round(theta, decimals or 2)
        return f"RBS({theta})"

"""

import pennylane as qml

class RBS(qml.operation.Operation):
    num_wires = 2
    num_params = 1
    grad_method = "A"
    name = "RBS"

    @staticmethod
    def decomposition(theta, wires):
        return [
            qml.CNOT(wires=wires),
            qml.RY(theta, wires=wires[0]),
            qml.RY(-theta, wires=wires[1]),
            qml.CNOT(wires=wires),
        ]

    def label(self, decimals=None, **kwargs):
        theta = qml.math.toarray(self.parameters[0]).item()
        theta = round(theta, decimals or 2)
        return f"RBS({theta})"


In [ ]:
def butterfly_layer(thetas, wires):
    """
    Orthogonal Butterfly Layer
    thetas: lista ou array de ângulos (um por RBS)
    wires: lista de qubits (tamanho N = 2^m)
    """
    N = len(wires)
    S = int(np.log2(N))
    idx = 0

    for s in range(S):
        d = 2 ** s
        for i in range(N):
            if (i % (2 * d)) < d:
                RBS(thetas[idx], wires=[wires[i], wires[i + d]])
                idx += 1


In [ ]:
n_qubits = 16
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def circuit(thetas):
    butterfly_layer(thetas, wires=list(range(n_qubits)))
    return qml.state()


In [ ]:
print(qml.draw(circuit, decimals=2)(thetas))

In [ ]:
import numpy as np

def reverse_butterfly_layer(thetas, wires):
    """
    Reverse Orthogonal Butterfly Layer usando RBS
    """
    N = len(wires)
    S = int(np.log2(N))
    idx = 0

    for s in reversed(range(S)):
        d = 2 ** s
        for i in range(N):
            if (i % (2 * d)) < d:
                RBS(thetas[idx], wires=[wires[i], wires[i + d]])
                idx += 1


In [ ]:
n_qubits = 16
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def reverse_butterfly_rbs_ansatz(thetas):
    reverse_butterfly_layer(thetas, wires=list(range(n_qubits)))
    return qml.state()


In [ ]:
thetas = np.linspace(0, 1, 32)
print(qml.draw(reverse_butterfly_rbs_ansatz, decimals=2)(thetas))

In [ ]:
import numpy as np
import pennylane as qml

def reverse_butterfly_layer(thetas, wires):
    N = len(wires)
    S = int(np.log2(N))
    idx = 0

    for s in reversed(range(S)):
        d = 2 ** s
        for i in range(N):
            if (i % (2 * d)) < d:
                RBS(thetas[idx], wires=[wires[i], wires[i + d]])
                idx += 1


In [ ]:
def reverse_butterfly_rbs_ansatz(thetas, wires):
    reverse_butterfly_layer(thetas, wires)


In [ ]:
n_qubits = 16
wires = list(range(n_qubits))
thetas = np.linspace(0, 1, 32)

with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)


In [ ]:
with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)


In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt

def RBS(theta, wires):
    i, j = wires
    qml.CNOT(wires=[i, j])
    qml.RY(theta, wires=j)
    qml.CNOT(wires=[i, j])

def reverse_butterfly_rbs_ansatz(thetas, wires):
    n = len(wires)
    idx = 0
    d = 1

    while d < n:
        for start in range(0, n, 2*d):
            for i in range(d):
                RBS(
                    thetas[idx],
                    wires=[wires[start + i], wires[start + i + d]]
                )
                idx += 1
        d *= 2


In [ ]:
n = 16
wires = list(range(n))
n_layers = int(np.log2(n))
n_rbs = sum(n // (2**(l+1)) for l in range(n_layers))
thetas = np.random.uniform(0, 2*np.pi, n_rbs)

with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)

fig, ax = qml.drawer.draw_mpl(
    tape,
    wire_order=wires,
    show_all_wires=True
)

fig.set_size_inches(18, 6)
plt.tight_layout()
plt.show()


In [ ]:
print(len(tape.operations))  # deve dar 32

In [ ]:
drawer = qml.drawer.MPLDrawer(
    wires=wires,
    decimals=2,
    show_all_wires=True,
)


In [ ]:
fig, ax = drawer.draw()
fig.set_size_inches(14, 6)


In [ ]:
n_qubits = 16
wires = list(range(n_qubits))
thetas = np.linspace(0, 1, 32)

with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)

drawer = qml.drawer.MPLDrawer(
    wires=wires,
    decimals=2,
    show_all_wires=True,
)

fig, ax = drawer.draw(tape)
fig.set_size_inches(14, 6)


In [ ]:
with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)


In [ ]:
drawer = qml.drawer.MPLDrawer()

In [ ]:
n_qubits = 16
wires = list(range(n_qubits))


In [ ]:
wire_map = {w: i for i, w in enumerate(wires)}

In [ ]:
n_layers = 4

In [ ]:
drawer = qml.drawer.MPLDrawer(
    n_layers=n_layers,
    wire_map=wire_map
)

In [ ]:
fig, ax = drawer.draw(
    tape,
    decimals=2,
    show_all_wires=True
)

fig.set_size_inches(14, 6)


In [ ]:
n_qubits = 16
wires = list(range(n_qubits))
thetas = np.linspace(0, 1, 32)

with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)

wire_map = {w: i for i, w in enumerate(wires)}
n_layers = 4  # reverse butterfly: d = 8,4,2,1

drawer = qml.drawer.MPLDrawer(
    n_layers=n_layers,
    wire_map=wire_map
)

fig, ax = drawer.draw(
    tape,
    decimals=2,
    show_all_wires=True
)

fig.set_size_inches(4, 3)


In [ ]:
n_qubits = 16
wires = list(range(n_qubits))
thetas = np.linspace(0, 1, 32)

with qml.tape.QuantumTape() as tape:
    reverse_butterfly_rbs_ansatz(thetas, wires)

wire_map = {w: i for i, w in enumerate(wires)}
n_layers = 4  # d = 8,4,2,1

drawer = qml.drawer.MPLDrawer(
    n_layers=n_layers,
    wire_map=wire_map
)

fig = drawer.draw(tape)
fig.set_size_inches(14, 6)
